In [0]:
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F
import pandas as pd

w = WorkspaceClient()

In [0]:
# Call the API
response = w.genie.list_spaces()

# Access the 'spaces' attribute inside the response object
if response.spaces:
    data = [{"Space Name": s.title, "Space ID": s.space_id} for s in response.spaces]
    display(pd.DataFrame(data))
else:
    print("No Genie spaces found or access denied.")

In [0]:
target_space_id = "01f107d4952d17669242a64d98ef35dd"

chat_history = []

conversations = w.genie.list_conversations(space_id=target_space_id, include_all=True)

for conv in conversations.conversations:
    messages_resp = w.genie.list_conversation_messages(
        space_id=target_space_id, 
        conversation_id=conv.conversation_id
    )
    
    for msg in messages_resp.messages:
        user_query = msg.content
        actual_answer = "N/A"
        genie_sql = "N/A"
        
        # Extracting both the SQL and the final English answer
        if msg.attachments:
            for att in msg.attachments:
                if att.query:
                    genie_sql = att.query.query
                if att.text:
                    actual_answer = att.text.content
        
        chat_history.append({
            "conversation_id": conv.conversation_id,
            "user_id": msg.user_id,
            "event_timestamp_ms": msg.created_timestamp,
            "user_question": user_query,
            "chatbot_response": actual_answer
        })

df = pd.DataFrame(chat_history)
display(df)

In [0]:
if not df.empty:
    # Create Spark DataFrame
    df = spark.createDataFrame(df)
    
    # Convert time and save to the 'default' schema
    df = df.withColumn("event_timestamp", F.from_unixtime(F.col("event_timestamp_ms") / 1000).cast("timestamp")) \
           .drop("event_timestamp_ms")
    
    # We use 'default' here because you cannot create new schemas
    df.write.mode("overwrite").saveAsTable("default.genie_conversations_history")
    print("Data saved successfully to table: default.genie_conversations_history")